# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col, length
from pyspark.sql.types import StringType

# Reading from bronze

In [0]:
df = spark.table('workspace.bronze.crm_sales_details')

# Remapping columns

In [0]:
RENAME_MAP = {
    'sls_ord_num': 'order_number',
    'sls_prd_key': 'product_key',
    'sls_cust_id': 'customer_id',
    'sls_order_dt': 'order_date',
    'sls_ship_dt': 'ship_date',
    'sls_due_dt': 'due_date',
    'sls_sales': 'sales',
    'sls_quantity': 'quantity',
    'sls_price': 'price'
}

In [0]:
for old_value, new_value in RENAME_MAP.items(): 
    df = df.withColumnRenamed(old_value, new_value)    

In [0]:
df.printSchema()

# Trimming Values

In [0]:
for i in df.schema.fields:
    # print(i.name, i.dataType)
    if isinstance(i.dataType, StringType):
        df = df.withColumn(i.name, trim(col(i.name)))

# Cleaning dates

In [0]:
df.limit(10).display()

In [0]:
df = (
        df.withColumn(
        'order_date',
        F.when(
            (col('order_date') == 0) | (length(col('order_date')) != 8), None
        ).otherwise(F.to_date(col('order_date').cast("String"), "yyyyMMdd"))
    )
        .withColumn(
        "ship_date",
        F.when(
            (col("ship_date") == 0) | (length(col("ship_date")) != 8),
            None
        ).otherwise(F.to_date(col("ship_date").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "due_date",
        F.when(
            (col("due_date") == 0) | (length(col("due_date")) != 8),
            None
        ).otherwise(F.to_date(col("due_date").cast("string"), "yyyyMMdd"))
    )
)

In [0]:
df.select('due_date').display()

# Sales and Price Correction

In [0]:
df = (
df.withColumn(
        "price",
        F.when(
            (col("price").isNull()) | (col("price") <= 0),
            F.when(
                col("quantity") != 0,
                col("quantity") / col("quantity")
            ).otherwise(None)
        ).otherwise(col("price"))
    )    
)

# Wrinting Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

# Check Silver table

In [0]:
%sql
select * 
from workspace.silver.crm_sales